# EDA & Feature Engineering

The capstone for this track. Notebooks 02 through 06 taught you the tools; this notebook runs them end-to-end on a single Fintech Bank dataset and hands the result off to `machine-learning/`.

The flow we follow is the standard EDA loop, applied in order.

1. **First look** — shape, types, missing values, basic numeric summary.
2. **Univariate distributions** — what shape is each column?
3. **Bivariate relationships** — how does each feature relate to the target?
4. **Correlation pruning** — which features are redundant?
5. **Cleaning** — impute missing, cap outliers.
6. **Feature engineering** — log transforms, ratios, categorical encoding.
7. **Handoff** — train/test split, ready for `machine-learning/`.

**The target** we will predict is `has_defaulted` — binary, whether a customer has defaulted on at least one loan. Fintech Bank cares about predicting this *before* a loan is issued; the same EDA workflow applies whether the data is synthetic or real.


## Build the dataset

For this end-to-end pass we synthesize a 500-customer dataset shaped like real Fintech Bank data — segments, incomes, balances, loan counts, card activity, tenure — plus a target column whose probability depends on income, loan count, and tenure. We also inject realistic mess: missing values in three columns and a few outlier balances.

The point of the synthetic data is **predictable patterns** for teaching. In real life the patterns are weaker and more confounded — but the workflow is identical.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)
n = 500

# Segment drives income
segment = rng.choice(["retail", "wealth"], n, p=[0.75, 0.25])
income = np.where(
    segment == "retail",
    rng.lognormal(10.7, 0.5, n),
    rng.lognormal(11.7, 0.5, n),
).astype(int)

account_balance = (income * rng.uniform(2, 30, n)).astype(int)
# A few high-net-worth outliers
outlier_idx = rng.choice(n, 5, replace=False)
account_balance[outlier_idx] = account_balance[outlier_idx] * 50

age = rng.integers(22, 65, n)
city = rng.choice(["Bengaluru", "Mumbai", "Hyderabad", "Chennai", "Pune"], n)
num_loans = rng.poisson(1.5, n)
num_card_txns_30d = rng.poisson(20, n)
months_with_bank = rng.integers(1, 120, n)

# Default target — driven by low income, many loans, short tenure
logit = (
    -3
    + (np.log(income) < 10.5).astype(int) * 1.5
    + (num_loans > 3).astype(int) * 1.2
    + (months_with_bank < 12).astype(int) * 1.0
    + rng.normal(0, 0.5, n)
)
default_prob = 1 / (1 + np.exp(-logit))
has_defaulted = (rng.uniform(0, 1, n) < default_prob).astype(int)

customers = pd.DataFrame({
    "customer_id": range(1001, 1001 + n),
    "age": age,
    "city": city,
    "segment": segment,
    "monthly_income": income,
    "account_balance": account_balance,
    "num_loans": num_loans,
    "num_card_txns_30d": num_card_txns_30d,
    "months_with_bank": months_with_bank,
    "has_defaulted": has_defaulted,
}).set_index("customer_id")

# Inject realistic missingness
customers.loc[rng.choice(customers.index, 25, replace=False), "monthly_income"] = np.nan
customers.loc[rng.choice(customers.index, 10, replace=False), "account_balance"] = np.nan
customers.loc[rng.choice(customers.index, 5, replace=False), "city"] = None

print(f"shape:         {customers.shape}")
print(f"default rate:  {customers['has_defaulted'].mean():.1%}")
customers.head()


## Step 1 — first look

The trio from notebook 01: `info()`, `describe()`, and `isna().sum()`. Run these the moment a new DataFrame lands. They tell you, in under five seconds, what you are working with — shape, dtypes, null counts, and the numeric shape of each column.


In [ ]:
customers.info()
print("\nnulls per column:")
print(customers.isna().sum())
print("\nnumeric summary:")
customers.describe().round(2)


## Step 2 — univariate distributions

For each numeric column, look at its distribution. For each categorical column, look at its counts. This is the visual analog of `describe()` — same information, different format, often more informative.

Check three things on every plot.

- Is the distribution **symmetric** or **skewed**? Skewed columns will want a log transform later.
- Are there visible **outliers** in the tails?
- For categoricals, is one category **dominant**? That may be a class-imbalance problem in the target, or a low-signal column for the features.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 9))

numeric_cols = ["age", "monthly_income", "account_balance", "num_loans", "num_card_txns_30d", "months_with_bank"]
for ax, col in zip(axes.flat[:6], numeric_cols):
    sns.histplot(customers[col].dropna(), bins=25, ax=ax)
    ax.set_title(col)

# Categorical counts in the bottom row
sns.countplot(data=customers, y="city", order=customers["city"].value_counts().index, ax=axes[2, 0])
axes[2, 0].set_title("city")
sns.countplot(data=customers, x="segment", ax=axes[2, 1])
axes[2, 1].set_title("segment")
sns.countplot(data=customers, x="has_defaulted", ax=axes[2, 2])
axes[2, 2].set_title("has_defaulted (target)")

fig.tight_layout()
plt.show()


## Step 3 — bivariate relationships with the target

The question that drives everything else: **which features differ between defaulters and non-defaulters?**

A feature whose distribution looks the same across the two target values has no predictive power. A feature whose distribution shifts is exactly what a model will learn from. A grid of boxplots — one per numeric feature, split by `has_defaulted` — gives you this answer in one glance.

For categorical features, the equivalent move is a chi-square test on the cross-tab (notebook 06).


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.flat, numeric_cols):
    sns.boxplot(data=customers, x="has_defaulted", y=col, ax=ax)
    ax.set_title(col)
fig.tight_layout()
plt.show()

# Expect to see: monthly_income and months_with_bank shift DOWN for defaulters;
# num_loans shifts UP; age and num_card_txns_30d look similar in both groups.


In [ ]:
# Is the default rate different across cities? Chi-square on the cross-tab.
city_default = pd.crosstab(customers["city"], customers["has_defaulted"])
print(city_default)

chi2, p, dof, _ = stats.chi2_contingency(city_default)
print(f"\nchi-square statistic: {chi2:.3f}")
print(f"p-value:              {p:.3f}")
print("default rate varies by city?", p < 0.05)


## Step 4 — correlation pruning

A correlation heatmap on the numeric columns answers two questions at once.

- **Redundancy** — pairs of features with `|r| > 0.9` are carrying the same signal. Keep one, drop the other before modeling.
- **Target relationship** — the row (or column) for `has_defaulted` tells you which features are linearly correlated with the outcome.

For non-linear relationships, fall back to Spearman (covered in notebook 06). For most numeric features in real data, the two agree closely — Pearson is the default first pass.


In [ ]:
numeric_for_corr = numeric_cols + ["has_defaulted"]
corr = customers[numeric_for_corr].corr().round(2)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation matrix — numeric features + target")
plt.show()

print("\n|correlation| with has_defaulted, sorted high to low:")
print(corr["has_defaulted"].drop("has_defaulted").abs().sort_values(ascending=False))


## Step 5 — cleaning

Two fixes, in order.

- **Missing values.** Fill `monthly_income` with the **per-segment median** — this preserves the wealth/retail income gap rather than blurring it. Fill `account_balance` with the global median. Fill `city` with the sentinel `"(unknown)"` so it survives one-hot encoding instead of disappearing.
- **Outliers.** Cap `account_balance` at the 99th percentile so the few high-net-worth customers don't dominate a regression's coefficients.

For tree-based models the outlier cap matters less. For linear models, k-nearest-neighbors, and most distance-based methods, it matters a lot.


In [ ]:
clean = customers.copy()

# Segment-aware median for income — preserves the segment gap
clean["monthly_income"] = clean["monthly_income"].fillna(
    clean.groupby("segment")["monthly_income"].transform("median")
)

# Global median for account_balance
clean["account_balance"] = clean["account_balance"].fillna(clean["account_balance"].median())

# Sentinel for missing city
clean["city"] = clean["city"].fillna("(unknown)")

# Cap account_balance at the 99th percentile
cap = clean["account_balance"].quantile(0.99)
clean["account_balance"] = clean["account_balance"].clip(upper=cap)

print("nulls remaining:")
print(clean.isna().sum())
print(f"\naccount_balance capped at {int(cap):,}")


## Step 6 — feature engineering

Three moves take this dataset from "clean" to "model-ready."

1. **Log-transform skewed numerics.** `monthly_income` and `account_balance` are right-skewed. Apply `np.log1p` (which is `log(1 + x)` and handles zeros gracefully) to make them roughly normal. Notebook 06 showed why most models prefer this.
2. **Ratio features.** Models pick up *relationships* between features, but you can give them a head start by hand-crafting the ratios you know matter. `num_loans / months_with_bank` is "loan acquisition rate" — a single number with more signal than either input alone. `account_balance / monthly_income` is "months of savings."
3. **One-hot encode categoricals.** `city` and `segment` are strings; models want numbers. `pd.get_dummies` creates one binary column per category value. `drop_first=True` drops one to avoid the dummy-variable trap (perfect multicollinearity).


In [ ]:
features = clean.copy()

# Log transforms on the skewed columns
features["log_income"] = np.log1p(features["monthly_income"])
features["log_balance"] = np.log1p(features["account_balance"])

# Ratio features — domain knowledge baked into the input
features["loans_per_month"] = features["num_loans"] / features["months_with_bank"].clip(lower=1)
features["balance_to_income_ratio"] = features["account_balance"] / features["monthly_income"]

# Drop the originals we replaced
features = features.drop(columns=["monthly_income", "account_balance"])

# One-hot encode categorical columns
features = pd.get_dummies(features, columns=["city", "segment"], drop_first=True)

print("final feature matrix shape:", features.shape)
print("\ncolumns:")
print(features.columns.tolist())
features.head()


In [ ]:
# Train / test split — stratify on the target so class balance survives the split
X = features.drop(columns=["has_defaulted"])
y = features["has_defaulted"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)

print(f"X_train: {X_train.shape}    y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}    y_test:  {y_test.shape}")
print(f"\ntrain default rate: {y_train.mean():.2%}")
print(f"test  default rate: {y_test.mean():.2%}")
print("(stratify=y keeps the class balance equal across train and test)")


## Handoff to `machine-learning/`

You now have four arrays — `X_train`, `X_test`, `y_train`, `y_test` — with:

- No missing values.
- No string columns — everything is numeric or boolean.
- No extreme outliers — capped at the 99th percentile.
- Skewed columns transformed to roughly normal.
- A target class balance preserved across train and test via stratified sampling.

This is exactly the input shape `sklearn`'s classifiers expect. In `machine-learning/` you would now fit a baseline model — logistic regression on the same features above is the right starting point — evaluate on the test set (precision, recall, ROC-AUC), and iterate with tree-based models, more features, and hyperparameter tuning.

### What this track gave you

| Notebook | What you can now do |
|---|---|
| 01 | Map any data work to the acquire → clean → explore → model → communicate loop |
| 02 | Reach for vectorized NumPy operations instead of writing loops |
| 03 | Build `Series` and `DataFrame`s, and read from CSV, JSON, parquet, and SQL |
| 04 | Handle missing data, `groupby`, `merge`, `pivot`/`melt`, and time-series resample |
| 05 | Build any figure-and-axes plot with matplotlib + seaborn |
| 06 | Test hypotheses, measure effect sizes, and read correlation honestly |
| 07 | Run a full EDA + feature engineering pass that hands clean data to ML |

The four pillars from `LEARNING_FRAMEWORK.md` — notebook, audio, NodeMap, Claude Q&A — apply to every concept ahead of you. The work from here is repetition: build a NodeMap that compresses each notebook into 5–7 nodes, listen to the audio on a commute, then re-derive the queries from memory a week later.

Onward to `machine-learning/`.
